### BASIC STATISTICS-2

#### Problem Statement: Hospital Patient Data Analysis

#### Context:

A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

#### Tasks

1. Load the patient dataset and show summary with info().

2. Select only the columns relevent for billing:      ['PatientID','Department', 'Doctor', 'BillAmount'].

3. Drop administrative columns like ['ReceptionistID','CheckInTime'].

4. Use groupby to find total bill amount per department.

5. Remove duplicate patient records based on PatientID.

6. Fill missing Bill Amount values with the mean bill amount.

7. Merge the billing dataset with patient dataset on PatientID.

8. Concatenate an additional DataFrame that contains new patients for the current week (rowwise).

9. Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (columnwise).

#### Expected Outcome:

·  Final cleaned dataset with accurate billing info.

·  All missing values handled, merged dataset across PatientID.

 · Ability to perform further analytics on department-wise revenue or doctor performance.

In [1]:
import pandas as pd

patient = pd.read_csv("C:/Users/adish/Downloads/Patient_Data.csv")
billing = pd.read_csv("C:/Users/adish/Downloads/Billing_Data.csv")


In [2]:
patient.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [3]:
billing.head()

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


#### 2. Select Columns Relevant for Billing

In [4]:
billing_cols = patient[['PatientID','Department','Doctor','BillAmount']]
print(billing_cols.head())


   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN


#### 3. Drop Administrative Columns

In [5]:
patient = patient.drop(['ReceptionistID','CheckInTime'], axis=1)


#### 4. Total Bill Amount per Department (groupby)

In [6]:
dept_bill = billing_cols.groupby('Department')['BillAmount'].sum()
print(dept_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


#### 5. Remove Duplicate Patient Records

In [7]:
patient = patient.drop_duplicates(subset='PatientID')


#### 6. Fill Missing Bill Amount with Mean

In [8]:
mean_bill = billing_cols['BillAmount'].mean()
billing_cols['BillAmount'].fillna(mean_bill, inplace=True)


C:\Users\adish\AppData\Local\Temp\ipykernel_31624\2358945033.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  billing_cols['BillAmount'].fillna(mean_bill, inplace=True)


0    5000.0
1    5925.0
2    7500.0
3    6200.0
4    5925.0
5    5000.0
Name: BillAmount, dtype: float64

#### 7. Merge Patient & Billing Data

In [9]:
merged_data = pd.merge(patient, billing, on='PatientID', how='inner')
print(merged_data.head())


   PatientID     Name   Department     Doctor  BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith      5000.0              2000   
1        102      Bob    Neurology   Dr. John         NaN              1500   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0              2500   
3        104    David   Cardiology  Dr. Smith      6200.0              3000   
4        105      Eva  Dermatology   Dr. Rose         NaN              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


#### 8. Concatenate New Patient Records (Row-wise)

In [10]:
new_patients = pd.DataFrame({
    'PatientID':[201,202],
    'Department':['Cardiology','Orthopedics'],
    'Doctor':['Dr A','Dr B']
})

patient = pd.concat([patient, new_patients], axis=0)


#### 9. Concatenate New Billing Columns (Column-wise)

In [12]:
billing_extra = pd.DataFrame({
    'InsuranceCovered':['Yes','No'],
    'FinalAmount':[4500,6200]
})

final_data = pd.concat([merged_data, billing_extra], axis=1)
print(final_data.head())


   PatientID     Name   Department     Doctor  BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith      5000.0              2000   
1        102      Bob    Neurology   Dr. John         NaN              1500   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0              2500   
3        104    David   Cardiology  Dr. Smith      6200.0              3000   
4        105      Eva  Dermatology   Dr. Rose         NaN              1000   

   FinalAmount InsuranceCovered  FinalAmount  
0         3000              Yes       4500.0  
1         3500               No       6200.0  
2         5000              NaN          NaN  
3         3200              NaN          NaN  
4         4000              NaN          NaN  


#### 10. Final Dataset

In [13]:
final_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         5 non-null      int64  
 1   Name              5 non-null      str    
 2   Department        5 non-null      str    
 3   Doctor            5 non-null      str    
 4   BillAmount        3 non-null      float64
 5   InsuranceCovered  5 non-null      int64  
 6   FinalAmount       5 non-null      int64  
 7   InsuranceCovered  2 non-null      str    
 8   FinalAmount       2 non-null      float64
dtypes: float64(2), int64(3), str(4)
memory usage: 492.0 bytes
